In [1]:
import torch
import torch.nn.functional as F
torch.set_printoptions(precision=2)


In [5]:
# ── Config ──────────────────────────────────────────────────────
T = 5                # tokens
d = 4                # model dimension
n_heads = 2          # query heads
n_kv_heads = 1       # KV heads (fewer = grouped/shared)
head_dim = 2         # dimension per head
n_rep = n_heads // n_kv_heads  # 2 query heads share each KV head
tokens = ["A", "B", "C", "D", "E"]

In [6]:
print("╔══════════════════════════════════════════════════╗")
print("║  GROUPED QUERY ATTENTION (GQA) — Toy Example    ║")
print("╚══════════════════════════════════════════════════╝")
print(f"""
  T = {T}  tokens
  d = {d}  model dimension
  n_heads    = {n_heads}  (query heads)
  n_kv_heads = {n_kv_heads}  (key/value heads)
  head_dim   = {head_dim}
  → Each KV head is shared by {n_rep} query heads
  → Wq shape: ({d}, {n_heads * head_dim})   ← projects to ALL query heads
  → Wk shape: ({d}, {n_kv_heads * head_dim})   ← projects to FEWER KV heads
  → Wv shape: ({d}, {n_kv_heads * head_dim})   ← projects to FEWER KV heads
  → Wo shape: ({n_heads * head_dim}, {d})   ← merges heads back to model dim
  In standard MHA:  Wk would be ({d}, {n_heads * head_dim}) — one K per Q head
  GQA saves memory: Wk is only  ({d}, {n_kv_heads * head_dim}) — shared across Q heads
""")
# ── Input ───────────────────────────────────────────────────────
X = torch.tensor([
    [1, 0, 2, 1],   # A
    [0, 1, 1, 0],   # B
    [2, 1, 0, 1],   # C
    [1, 2, 1, 0],   # D
    [0, 0, 2, 1],   # E
], dtype=torch.float)
# Wq: (d, n_heads * head_dim) = (4, 4)
Wq = torch.tensor([
    [1, 0, 0, 1],
    [0, 1, 1, 0],
    [1, 1, 0, 0],
    [0, 0, 1, 1],
], dtype=torch.float)
# Wk: (d, n_kv_heads * head_dim) = (4, 2)  ← SMALLER than Wq
Wk = torch.tensor([
    [1, 0],
    [0, 1],
    [1, 1],
    [0, 1],
], dtype=torch.float)
# Wv: (d, n_kv_heads * head_dim) = (4, 2)  ← SMALLER than Wq
Wv = torch.tensor([
    [0, 1],
    [1, 0],
    [1, 1],
    [0, 1],
], dtype=torch.float)
# Wo: (n_heads * head_dim, d) = (4, 4) — output projection
Wo = torch.eye(d)
# FFN weights (simple)
Wff_up   = torch.ones(d, d) * 0.5
Wff_down = torch.ones(d, d) * 0.5

╔══════════════════════════════════════════════════╗
║  GROUPED QUERY ATTENTION (GQA) — Toy Example    ║
╚══════════════════════════════════════════════════╝

  T = 5  tokens
  d = 4  model dimension
  n_heads    = 2  (query heads)
  n_kv_heads = 1  (key/value heads)
  head_dim   = 2
  → Each KV head is shared by 2 query heads
  → Wq shape: (4, 4)   ← projects to ALL query heads
  → Wk shape: (4, 2)   ← projects to FEWER KV heads
  → Wv shape: (4, 2)   ← projects to FEWER KV heads
  → Wo shape: (4, 4)   ← merges heads back to model dim
  In standard MHA:  Wk would be (4, 4) — one K per Q head
  GQA saves memory: Wk is only  (4, 2) — shared across Q heads



In [7]:
def print_matrix(label, mat, row_labels=None):
    print(f"\n── {label}  (shape {list(mat.shape)}) ──")
    if row_labels is None:
        row_labels = [str(i) for i in range(mat.shape[0])]
    cols = [f"d{i}" for i in range(mat.shape[-1])]
    print(f"  {'':4} {cols}")
    for i, lbl in enumerate(row_labels):
        print(f"  {lbl:4} {[round(x, 2) for x in mat[i].tolist()]}")


In [8]:
def gqa_attention(X, Wq, Wk, Wv, Wo):
    """GQA attention with full trace."""
    T, d = X.shape
    scale = head_dim ** 0.5
    # ── Step 1: Project ─────────────────────────────────────────
    Q_full = X @ Wq     # (T, n_heads * head_dim) = (5, 4)
    K_full = X @ Wk     # (T, n_kv_heads * head_dim) = (5, 2)
    V_full = X @ Wv     # (T, n_kv_heads * head_dim) = (5, 2)
    print_matrix("Q_full  (X @ Wq)", Q_full, tokens)
    print_matrix("K_full  (X @ Wk)  ← only 2 cols, not 4!", K_full, tokens)
    print_matrix("V_full  (X @ Wv)  ← only 2 cols, not 4!", V_full, tokens)
    # ── Step 2: Reshape into heads ──────────────────────────────
    Q = Q_full.view(T, n_heads, head_dim)       # (5, 2, 2)
    K = K_full.view(T, n_kv_heads, head_dim)    # (5, 1, 2)
    V = V_full.view(T, n_kv_heads, head_dim)    # (5, 1, 2)
    print(f"\n── Reshape into heads ──")
    print(f"  Q reshaped: {list(Q.shape)}  →  {n_heads} query heads, each dim {head_dim}")
    print(f"  K reshaped: {list(K.shape)}  →  {n_kv_heads} KV head(s), each dim {head_dim}")
    print(f"  V reshaped: {list(V.shape)}  →  {n_kv_heads} KV head(s), each dim {head_dim}")
    for h in range(n_heads):
        print(f"\n  Q head {h}:")
        for i, tok in enumerate(tokens):
            print(f"    {tok}: {Q[i, h].tolist()}")
    for h in range(n_kv_heads):
        print(f"\n  K head {h}  (shared by Q heads {list(range(h * n_rep, (h+1) * n_rep))}):")
        for i, tok in enumerate(tokens):
            print(f"    {tok}: {K[i, h].tolist()}")
        print(f"\n  V head {h}  (shared by Q heads {list(range(h * n_rep, (h+1) * n_rep))}):")
        for i, tok in enumerate(tokens):
            print(f"    {tok}: {V[i, h].tolist()}")
    # ── Step 3: Expand K, V to match Q head count ───────────────
    # This is the KEY GQA operation: replicate each KV head n_rep times
    K_expanded = K.repeat(1, n_rep, 1)   # (5, 1, 2) → (5, 2, 2)
    V_expanded = V.repeat(1, n_rep, 1)   # (5, 1, 2) → (5, 2, 2)
    print(f"\n── Expand K, V  (repeat each KV head {n_rep}× to match {n_heads} Q heads) ──")
    print(f"  K: {list(K.shape)} → {list(K_expanded.shape)}")
    print(f"  V: {list(V.shape)} → {list(V_expanded.shape)}")
    print(f"  ⚡ This is where GQA saves memory vs MHA:")
    print(f"     MHA would store {n_heads} separate K,V heads = {n_heads * T * head_dim} values")
    print(f"     GQA only stores {n_kv_heads} K,V head(s)    = {n_kv_heads * T * head_dim} values")
    # ── Step 4: Compute attention per head ──────────────────────
    # Transpose to (n_heads, T, head_dim) for batched matmul
    Q_t = Q.permute(1, 0, 2)             # (n_heads, T, head_dim)
    K_t = K_expanded.permute(1, 0, 2)    # (n_heads, T, head_dim)
    V_t = V_expanded.permute(1, 0, 2)    # (n_heads, T, head_dim)
    scores = (Q_t @ K_t.transpose(-2, -1)) / scale   # (n_heads, T, T)
    # Causal mask
    causal_mask = torch.triu(torch.full((T, T), float('-inf')), diagonal=1)
    print(f"\n── Causal Mask ──")
    print(f"  query↓  key→  {tokens}")
    for i, tok in enumerate(tokens):
        row = ["  ✓" if causal_mask[i, j] == 0 else "  ✗" for j in range(T)]
        print(f"  {tok}          {''.join(row)}")
    scores_masked = scores + causal_mask    # broadcast across heads
    for h in range(n_heads):
        kv_idx = h // n_rep
        print(f"\n── Head {h}  (uses KV head {kv_idx}) ──")
        print_matrix(f"  Raw scores Q{h} @ K{kv_idx}ᵀ / √d", scores[h], tokens)
        print_matrix(f"  Masked scores", scores_masked[h], tokens)
    attn_weights = torch.softmax(scores_masked, dim=-1)   # (n_heads, T, T)
    for h in range(n_heads):
        kv_idx = h // n_rep
        print_matrix(f"Head {h} attention weights  (softmax)", attn_weights[h], tokens)
    # ── Step 5: Weighted sum of values ──────────────────────────
    head_outputs = attn_weights @ V_t    # (n_heads, T, head_dim)
    for h in range(n_heads):
        print_matrix(f"Head {h} output  (weights @ V)", head_outputs[h], tokens)
    # ── Step 6: Concatenate heads + output projection ───────────
    # (n_heads, T, head_dim) → (T, n_heads, head_dim) → (T, n_heads * head_dim)
    concat = head_outputs.permute(1, 0, 2).contiguous().view(T, n_heads * head_dim)
    print_matrix("Concatenated heads", concat, tokens)
    out = concat @ Wo     # (T, d)
    print_matrix("After output projection  (concat @ Wo)", out, tokens)
    return out

In [9]:

def transformer_block(x, Wq, Wk, Wv, Wo, Wff_up, Wff_down, block_name="Block"):
    print(f"\n{'█' * 55}")
    print(f"  {block_name}")
    print(f"{'█' * 55}")
    print_matrix("Input to block", x, tokens)
    # 1. Attention
    print(f"\n┌─ GQA Attention sublayer {'─' * 28}")
    attn_out = gqa_attention(x, Wq, Wk, Wv, Wo)
    # 2. Residual + LayerNorm
    x = F.layer_norm(x + attn_out, [x.shape[-1]])
    print_matrix("After  attn + residual + LayerNorm", x, tokens)
    # 3. FFN
    print(f"\n┌─ FFN sublayer {'─' * 37}")
    ffn_out = F.relu(x @ Wff_up) @ Wff_down
    print_matrix("FFN output", ffn_out, tokens)
    # 4. Residual + LayerNorm
    x = F.layer_norm(x + ffn_out, [x.shape[-1]])
    print_matrix("After  FFN + residual + LayerNorm  ← enters next block", x, tokens)
    return x

In [10]:
# ── Run ─────────────────────────────────────────────────────────
H1 = transformer_block(X, Wq, Wk, Wv, Wo, Wff_up, Wff_down,
                        block_name="BLOCK 1 — GQA Attention")


███████████████████████████████████████████████████████
  BLOCK 1 — GQA Attention
███████████████████████████████████████████████████████

── Input to block  (shape [5, 4]) ──
       ['d0', 'd1', 'd2', 'd3']
  A    [1.0, 0.0, 2.0, 1.0]
  B    [0.0, 1.0, 1.0, 0.0]
  C    [2.0, 1.0, 0.0, 1.0]
  D    [1.0, 2.0, 1.0, 0.0]
  E    [0.0, 0.0, 2.0, 1.0]

┌─ GQA Attention sublayer ────────────────────────────

── Q_full  (X @ Wq)  (shape [5, 4]) ──
       ['d0', 'd1', 'd2', 'd3']
  A    [3.0, 2.0, 1.0, 2.0]
  B    [1.0, 2.0, 1.0, 0.0]
  C    [2.0, 1.0, 2.0, 3.0]
  D    [2.0, 3.0, 2.0, 1.0]
  E    [2.0, 2.0, 1.0, 1.0]

── K_full  (X @ Wk)  ← only 2 cols, not 4!  (shape [5, 2]) ──
       ['d0', 'd1']
  A    [3.0, 3.0]
  B    [1.0, 2.0]
  C    [2.0, 2.0]
  D    [2.0, 3.0]
  E    [2.0, 3.0]

── V_full  (X @ Wv)  ← only 2 cols, not 4!  (shape [5, 2]) ──
       ['d0', 'd1']
  A    [2.0, 4.0]
  B    [2.0, 1.0]
  C    [1.0, 3.0]
  D    [3.0, 2.0]
  E    [2.0, 3.0]

── Reshape into heads ──
  Q reshape

In [11]:
# ---All in one-----
import torch
import torch.nn.functional as F
torch.set_printoptions(precision=2)

# ── Config ──────────────────────────────────────────────────────
T = 5                # tokens
d = 4                # model dimension
n_heads = 2          # query heads
n_kv_heads = 1       # KV heads (fewer = grouped/shared)
head_dim = 2         # dimension per head
n_rep = n_heads // n_kv_heads  # 2 query heads share each KV head

tokens = ["A", "B", "C", "D", "E"]

print("╔══════════════════════════════════════════════════╗")
print("║  GROUPED QUERY ATTENTION (GQA) — Toy Example    ║")
print("╚══════════════════════════════════════════════════╝")
print(f"""
  T = {T}  tokens
  d = {d}  model dimension
  n_heads    = {n_heads}  (query heads)
  n_kv_heads = {n_kv_heads}  (key/value heads)
  head_dim   = {head_dim}

  → Each KV head is shared by {n_rep} query heads
  → Wq shape: ({d}, {n_heads * head_dim})   ← projects to ALL query heads
  → Wk shape: ({d}, {n_kv_heads * head_dim})   ← projects to FEWER KV heads
  → Wv shape: ({d}, {n_kv_heads * head_dim})   ← projects to FEWER KV heads
  → Wo shape: ({n_heads * head_dim}, {d})   ← merges heads back to model dim

  In standard MHA:  Wk would be ({d}, {n_heads * head_dim}) — one K per Q head
  GQA saves memory: Wk is only  ({d}, {n_kv_heads * head_dim}) — shared across Q heads
""")

# ── Input ───────────────────────────────────────────────────────
X = torch.tensor([
    [1, 0, 2, 1],   # A
    [0, 1, 1, 0],   # B
    [2, 1, 0, 1],   # C
    [1, 2, 1, 0],   # D
    [0, 0, 2, 1],   # E
], dtype=torch.float)

# Wq: (d, n_heads * head_dim) = (4, 4)
Wq = torch.tensor([
    [1, 0, 0, 1],
    [0, 1, 1, 0],
    [1, 1, 0, 0],
    [0, 0, 1, 1],
], dtype=torch.float)

# Wk: (d, n_kv_heads * head_dim) = (4, 2)  ← SMALLER than Wq
Wk = torch.tensor([
    [1, 0],
    [0, 1],
    [1, 1],
    [0, 1],
], dtype=torch.float)

# Wv: (d, n_kv_heads * head_dim) = (4, 2)  ← SMALLER than Wq
Wv = torch.tensor([
    [0, 1],
    [1, 0],
    [1, 1],
    [0, 1],
], dtype=torch.float)

# Wo: (n_heads * head_dim, d) = (4, 4) — output projection
Wo = torch.eye(d)

# FFN weights (simple)
Wff_up   = torch.ones(d, d) * 0.5
Wff_down = torch.ones(d, d) * 0.5


def print_matrix(label, mat, row_labels=None):
    print(f"\n── {label}  (shape {list(mat.shape)}) ──")
    if row_labels is None:
        row_labels = [str(i) for i in range(mat.shape[0])]
    cols = [f"d{i}" for i in range(mat.shape[-1])]
    print(f"  {'':4} {cols}")
    for i, lbl in enumerate(row_labels):
        print(f"  {lbl:4} {[round(x, 2) for x in mat[i].tolist()]}")


def gqa_attention(X, Wq, Wk, Wv, Wo):
    """GQA attention with full trace."""
    T, d = X.shape
    scale = head_dim ** 0.5

    # ── Step 1: Project ─────────────────────────────────────────
    Q_full = X @ Wq     # (T, n_heads * head_dim) = (5, 4)
    K_full = X @ Wk     # (T, n_kv_heads * head_dim) = (5, 2)
    V_full = X @ Wv     # (T, n_kv_heads * head_dim) = (5, 2)

    print_matrix("Q_full  (X @ Wq)", Q_full, tokens)
    print_matrix("K_full  (X @ Wk)  ← only 2 cols, not 4!", K_full, tokens)
    print_matrix("V_full  (X @ Wv)  ← only 2 cols, not 4!", V_full, tokens)

    # ── Step 2: Reshape into heads ──────────────────────────────
    Q = Q_full.view(T, n_heads, head_dim)       # (5, 2, 2)
    K = K_full.view(T, n_kv_heads, head_dim)    # (5, 1, 2)
    V = V_full.view(T, n_kv_heads, head_dim)    # (5, 1, 2)

    print(f"\n── Reshape into heads ──")
    print(f"  Q reshaped: {list(Q.shape)}  →  {n_heads} query heads, each dim {head_dim}")
    print(f"  K reshaped: {list(K.shape)}  →  {n_kv_heads} KV head(s), each dim {head_dim}")
    print(f"  V reshaped: {list(V.shape)}  →  {n_kv_heads} KV head(s), each dim {head_dim}")

    for h in range(n_heads):
        print(f"\n  Q head {h}:")
        for i, tok in enumerate(tokens):
            print(f"    {tok}: {Q[i, h].tolist()}")

    for h in range(n_kv_heads):
        print(f"\n  K head {h}  (shared by Q heads {list(range(h * n_rep, (h+1) * n_rep))}):")
        for i, tok in enumerate(tokens):
            print(f"    {tok}: {K[i, h].tolist()}")
        print(f"\n  V head {h}  (shared by Q heads {list(range(h * n_rep, (h+1) * n_rep))}):")
        for i, tok in enumerate(tokens):
            print(f"    {tok}: {V[i, h].tolist()}")

    # ── Step 3: Expand K, V to match Q head count ───────────────
    # This is the KEY GQA operation: replicate each KV head n_rep times
    K_expanded = K.repeat(1, n_rep, 1)   # (5, 1, 2) → (5, 2, 2)
    V_expanded = V.repeat(1, n_rep, 1)   # (5, 1, 2) → (5, 2, 2)

    print(f"\n── Expand K, V  (repeat each KV head {n_rep}× to match {n_heads} Q heads) ──")
    print(f"  K: {list(K.shape)} → {list(K_expanded.shape)}")
    print(f"  V: {list(V.shape)} → {list(V_expanded.shape)}")
    print(f"  ⚡ This is where GQA saves memory vs MHA:")
    print(f"     MHA would store {n_heads} separate K,V heads = {n_heads * T * head_dim} values")
    print(f"     GQA only stores {n_kv_heads} K,V head(s)    = {n_kv_heads * T * head_dim} values")

    # ── Step 4: Compute attention per head ──────────────────────
    # Transpose to (n_heads, T, head_dim) for batched matmul
    Q_t = Q.permute(1, 0, 2)             # (n_heads, T, head_dim)
    K_t = K_expanded.permute(1, 0, 2)    # (n_heads, T, head_dim)
    V_t = V_expanded.permute(1, 0, 2)    # (n_heads, T, head_dim)

    scores = (Q_t @ K_t.transpose(-2, -1)) / scale   # (n_heads, T, T)

    # Causal mask
    causal_mask = torch.triu(torch.full((T, T), float('-inf')), diagonal=1)

    print(f"\n── Causal Mask ──")
    print(f"  query↓  key→  {tokens}")
    for i, tok in enumerate(tokens):
        row = ["  ✓" if causal_mask[i, j] == 0 else "  ✗" for j in range(T)]
        print(f"  {tok}          {''.join(row)}")

    scores_masked = scores + causal_mask    # broadcast across heads

    for h in range(n_heads):
        kv_idx = h // n_rep
        print(f"\n── Head {h}  (uses KV head {kv_idx}) ──")
        print_matrix(f"  Raw scores Q{h} @ K{kv_idx}ᵀ / √d", scores[h], tokens)
        print_matrix(f"  Masked scores", scores_masked[h], tokens)

    attn_weights = torch.softmax(scores_masked, dim=-1)   # (n_heads, T, T)

    for h in range(n_heads):
        kv_idx = h // n_rep
        print_matrix(f"Head {h} attention weights  (softmax)", attn_weights[h], tokens)

    # ── Step 5: Weighted sum of values ──────────────────────────
    head_outputs = attn_weights @ V_t    # (n_heads, T, head_dim)

    for h in range(n_heads):
        print_matrix(f"Head {h} output  (weights @ V)", head_outputs[h], tokens)

    # ── Step 6: Concatenate heads + output projection ───────────
    # (n_heads, T, head_dim) → (T, n_heads, head_dim) → (T, n_heads * head_dim)
    concat = head_outputs.permute(1, 0, 2).contiguous().view(T, n_heads * head_dim)
    print_matrix("Concatenated heads", concat, tokens)

    out = concat @ Wo     # (T, d)
    print_matrix("After output projection  (concat @ Wo)", out, tokens)

    return out


def transformer_block(x, Wq, Wk, Wv, Wo, Wff_up, Wff_down, block_name="Block"):
    print(f"\n{'█' * 55}")
    print(f"  {block_name}")
    print(f"{'█' * 55}")

    print_matrix("Input to block", x, tokens)

    # 1. Attention
    print(f"\n┌─ GQA Attention sublayer {'─' * 28}")
    attn_out = gqa_attention(x, Wq, Wk, Wv, Wo)

    # 2. Residual + LayerNorm
    x = F.layer_norm(x + attn_out, [x.shape[-1]])
    print_matrix("After  attn + residual + LayerNorm", x, tokens)

    # 3. FFN
    print(f"\n┌─ FFN sublayer {'─' * 37}")
    ffn_out = F.relu(x @ Wff_up) @ Wff_down
    print_matrix("FFN output", ffn_out, tokens)

    # 4. Residual + LayerNorm
    x = F.layer_norm(x + ffn_out, [x.shape[-1]])
    print_matrix("After  FFN + residual + LayerNorm  ← enters next block", x, tokens)

    return x


# ── Run ─────────────────────────────────────────────────────────
H1 = transformer_block(X, Wq, Wk, Wv, Wo, Wff_up, Wff_down,
                        block_name="BLOCK 1 — GQA Attention")


╔══════════════════════════════════════════════════╗
║  GROUPED QUERY ATTENTION (GQA) — Toy Example    ║
╚══════════════════════════════════════════════════╝

  T = 5  tokens
  d = 4  model dimension
  n_heads    = 2  (query heads)
  n_kv_heads = 1  (key/value heads)
  head_dim   = 2

  → Each KV head is shared by 2 query heads
  → Wq shape: (4, 4)   ← projects to ALL query heads
  → Wk shape: (4, 2)   ← projects to FEWER KV heads
  → Wv shape: (4, 2)   ← projects to FEWER KV heads
  → Wo shape: (4, 4)   ← merges heads back to model dim

  In standard MHA:  Wk would be (4, 4) — one K per Q head
  GQA saves memory: Wk is only  (4, 2) — shared across Q heads


███████████████████████████████████████████████████████
  BLOCK 1 — GQA Attention
███████████████████████████████████████████████████████

── Input to block  (shape [5, 4]) ──
       ['d0', 'd1', 'd2', 'd3']
  A    [1.0, 0.0, 2.0, 1.0]
  B    [0.0, 1.0, 1.0, 0.0]
  C    [2.0, 1.0, 0.0, 1.0]
  D    [1.0, 2.0, 1.0, 0.0]
  E    [0.0, 